In [8]:
import sys, importlib, time 
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, 
                             confusion_matrix, ConfusionMatrixDisplay, 
                             classification_report)

PROJECT_ROOT=Path.cwd()
if PROJECT_ROOT.name=="notebooks":
    PROJECT_ROOT=PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT/"src"))

import data_loader,models,training,evaluation
for m in (data_loader,models,training,evaluation):
    importlib.reload(m)
from data_loader import create_binary_label
from models import ANNModel
from models import LSTMModel
from training import train_model
from evaluation import predict_model

import time

Data_path=r"C:/Users/urvi_/Downloads/dataset.h5"
RESULTS =PROJECT_ROOT / "results"
figs=RESULTS/"figures"
figs.mkdir(parents=True,exist_ok=True)
cache=RESULTS/"features_all_days.npz"

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device",device)

Device cuda


In [9]:
d=np.load(cache)
x_all=d["X"]
y_all=create_binary_label(d["y"],threshold=1000)
dt=pd.to_datetime(d["dt"])
day_of=np.array(dt.date)
t=np.round((dt-dt[0]).total_seconds().to_numpy()).astype(np.int64)
print(t[:6])
gaps,counts=np.unique(np.diff(t),return_counts=True)

seq=5
ends=np.array([i for i in range(seq-1,len(t))
               if t[i]-t[i-seq+1]==10*(seq-1)
               and day_of[i-seq+1]==day_of[i]])

idx_matrix=ends[:,None]-np.arange(seq-1,-1,-1)
print("valid sequences:",len(ends),"of",len(t))
print("idx_matrix:",idx_matrix.shape,"| first row: ",idx_matrix[0])

[ 0 10 20 30 40 50]
valid sequences: 73694 of 74771
idx_matrix: (73694, 5) | first row:  [0 1 2 3 4]


In [10]:
from typing import Any


days_sorted=sorted(set(day_of))
seq_day=day_of[ends]
seq_y=y_all[ends]

class SeqDataset(Dataset):
    def __init__(self,X,y):
        self.X=torch.tensor(X,dtype=torch.float32)
        self.y=torch.tensor(y,dtype=torch.float32)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, index):
        return self.X[index],self.y[index]
    
def run_fold_lstm(test_day,epochs=100,lr=1e-3,batch=64,hidden=64,
                  dropout=0.3,wd=1e-4,seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    i=days_sorted.index(test_day)
    val_day=days_sorted[(i+1)%len(days_sorted)]
    
    test_s=np.where(seq_day==test_day)[0]
    val_s=np.where(seq_day==val_day)[0]
    fit_s=np.where((seq_day!=test_day)&(seq_day!=val_day))[0]
    
    fit_rows=np.unique(idx_matrix[fit_s].ravel())
    ss=StandardScaler().fit(x_all[fit_rows])
    x_scaled=ss.transform(x_all).astype(np.float32)
    
    def make_loader(sel,shuffle):
        ds=SeqDataset(x_scaled[idx_matrix[sel]],seq_y[sel])
        return DataLoader(ds,batch_size=batch,shuffle=shuffle)

    train_loader=make_loader(fit_s,True)
    val_loader=make_loader(val_s,False)
    test_loader=make_loader(test_s,False)
    
    n_far,n_near=(seq_y[fit_s]==0).sum(),(seq_y[fit_s]==1).sum()
    criterion=nn.BCEWithLogitsLoss(pos_weight=torch.tensor([n_far/n_near],
            dtype=torch.float32).to(device))
    model=LSTMModel(input_size=100,hidden_size=hidden,dropout=dropout).to(device)
    optimizer=torch.optim.Adam(model.parameters(),lr=lr,weight_decay=wd)
    
    history=train_model(model,train_loader,criterion,optimizer,device,
                        epochs=epochs,val_loader=val_loader,patience=10)
    
    y_true,y_pred,y_prob=predict_model(model,test_loader,device)
    y_true=y_true.astype(int)
    y_pred=y_pred.astype(int).flatten()
    y_prob=y_prob.flatten()
    
    far_f1,near_f1=f1_score(y_true,y_pred,average=None,zero_division=0)
    return {
        "test_day":str(test_day),
        "val_day":str(val_day),
        "n_train":len(fit_s),
        "n_val":len(val_s),
        "n_test":len(test_s),
        "best_epoch":int(np.argmin(history["val_loss"]))+1,
        "epochs_run":len(history["train_loss"]),
        "accuracy":accuracy_score(y_true,y_pred),
        "weighted_f1":f1_score(y_true,y_pred,average="weighted",zero_division=0),
        "far_f1":far_f1,
        "near_f1":near_f1,
        "auc":roc_auc_score(y_true,y_prob)
    }, history, (y_true,y_pred,y_prob)
    

In [11]:
t0=time.time()
r,h,preds=run_fold_lstm(days_sorted[8])
print({k:(round(v,4) if isinstance(v,float) else v) for k,v in r.items()})
print("minutes:", round((time.time()-t0)/60,1))

Epoch 1/100 | Train loss: 0.4284 | Val loss: 0.2653
Epoch 2/100 | Train loss: 0.3712 | Val loss: 0.2555
Epoch 3/100 | Train loss: 0.3454 | Val loss: 0.2605
Epoch 4/100 | Train loss: 0.3194 | Val loss: 0.2708
Epoch 5/100 | Train loss: 0.2927 | Val loss: 0.2925
Epoch 6/100 | Train loss: 0.2712 | Val loss: 0.2937
Epoch 7/100 | Train loss: 0.2535 | Val loss: 0.3003
Epoch 8/100 | Train loss: 0.2373 | Val loss: 0.3259
Epoch 9/100 | Train loss: 0.2242 | Val loss: 0.3066
Epoch 10/100 | Train loss: 0.2136 | Val loss: 0.4038
Epoch 11/100 | Train loss: 0.2030 | Val loss: 0.3423
Epoch 12/100 | Train loss: 0.1926 | Val loss: 0.3469
Early stopping at epoch 12
{'test_day': '2023-06-24', 'val_day': '2023-06-25', 'n_train': 60106, 'n_val': 5072, 'n_test': 8516, 'best_epoch': 2, 'epochs_run': 12, 'accuracy': 0.888, 'weighted_f1': 0.8884, 'far_f1': np.float64(0.9131), 'near_f1': np.float64(0.8423), 'auc': 0.9462}
minutes: 2.0


**Running all 10 folds**

In [12]:
results_dir=PROJECT_ROOT/"results"
results, histories, pooled=[],{},[]
t_all=time.time()

for day in days_sorted:
    print(f"\n === Test Day {day} ===")
    r,h,preds=run_fold_lstm(day)
    histories[str(day)]=h
    pooled.append((str(day),*preds))
    pd.DataFrame(results).to_csv(results_dir/"lstm_10day_partial.csv",index=False)
    print({k:(round(v,4) if isinstance(v,float) else v) for k,v in r.items()})
    
print("\n Total minutes: ",round((time.time()-t_all)/60,1))


 === Test Day 2023-06-16 ===
Epoch 1/100 | Train loss: 0.4147 | Val loss: 0.4207
Epoch 2/100 | Train loss: 0.3467 | Val loss: 0.4301
Epoch 3/100 | Train loss: 0.3126 | Val loss: 0.5088
Epoch 4/100 | Train loss: 0.2829 | Val loss: 0.4767
Epoch 5/100 | Train loss: 0.2553 | Val loss: 0.5363
Epoch 6/100 | Train loss: 0.2378 | Val loss: 0.5330
Epoch 7/100 | Train loss: 0.2173 | Val loss: 0.5697
Epoch 8/100 | Train loss: 0.2004 | Val loss: 0.6577
Epoch 9/100 | Train loss: 0.1877 | Val loss: 0.6384
Epoch 10/100 | Train loss: 0.1781 | Val loss: 0.6213
Epoch 11/100 | Train loss: 0.1653 | Val loss: 0.7232
Early stopping at epoch 11
{'test_day': '2023-06-16', 'val_day': '2023-06-17', 'n_train': 57372, 'n_val': 8516, 'n_test': 7806, 'best_epoch': 1, 'epochs_run': 11, 'accuracy': 0.8573, 'weighted_f1': 0.8587, 'far_f1': np.float64(0.8856), 'near_f1': np.float64(0.8104), 'auc': 0.9306}

 === Test Day 2023-06-17 ===
Epoch 1/100 | Train loss: 0.4169 | Val loss: 0.3937
Epoch 2/100 | Train loss: 0.3510

In [23]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

p = pd.read_csv(results_dir / "ann_10day_predictions.csv")     # columns: test_day, true, pred, prob

def smooth(df, w):      # average of the last w recordings only, so it could run live
    return df.groupby("test_day")["prob"].transform(lambda s: s.rolling(w, min_periods=1).mean())

rows = []
for w in [1, 3, 5]:
    pr = smooth(p, w)
    pred = (pr >= 0.5).astype(int)
    rows.append({"window": w,
                 "accuracy": accuracy_score(p["true"], pred),
                 "weighted_f1": f1_score(p["true"], pred, average="weighted"),
                 "near_f1": f1_score(p["true"], pred),
                 "auc": roc_auc_score(p["true"], pr)})
print(pd.DataFrame(rows).round(4))

   window  accuracy  weighted_f1  near_f1     auc
0       1    0.8807       0.8812   0.8192  0.9336
1       3    0.8844       0.8848   0.8240  0.9381
2       5    0.8812       0.8815   0.8191  0.9365


In [20]:
p = pd.read_csv(results_dir / "ann_10day_predictions.csv")
print(p.shape)
print(p.columns.tolist())
assert "prob" in p.columns, "wrong file: this is not the predictions file"

(74771, 4)
['test_day', 'true', 'pred', 'prob']


In [29]:
Data_path=r"C:/Users/urvi_/Downloads/dataset.h5"
def build_feature_cache_v2(data_path, out_path, chunk=500, bad=(59, 60, 61)):
    with h5py.File(data_path, "r") as f:
        n, n_ch, n_b = f["X"].shape
        keep = [c for c in range(n_ch) if c not in bad]
        feats = np.empty((n, 3 * n_b), dtype=np.float32)
        for s in range(0, n, chunk):
            block = f["X"][s:s + chunk][:, keep, :]
            parts = [block.max(axis=1), block.mean(axis=1), np.percentile(block, 90, axis=1)]
            feats[s:s + chunk] = np.log10(np.concatenate(parts, axis=1) + 1e-20)
        y = f["y"][:]
        dt = np.array([x.decode() if isinstance(x, bytes) else str(x) for x in f["datetimes"][:]])
    np.savez(out_path, X=feats, y=y, dt=dt)

build_feature_cache_v2(Data_path, results_dir / "features_all_days_v2.npz")

In [30]:
import os
p2 = results_dir / "features_all_days_v2.npz"
print(p2.exists(), round(os.path.getsize(p2) / 1e6, 1), "MB" if p2.exists() else "")
d2 = np.load(p2)
print(d2["X"].shape, d2["y"].shape)

True 97.8 MB
(74771, 300) (74771,)
